# Real-model evaluation in Google Colab

This notebook produces **experimental real-model evidence**, not production benchmarks. Colab hardware is ephemeral and can vary by session. Results apply only to the recorded model revision, tokenizer revision, decoding configuration, prompt, dataset, and runtime. Do not commit tokens, secrets, private datasets, downloaded model caches, or large generated runs. Use public-safe fixtures unless you have an approved evaluation protocol.

In [ ]:
import importlib.metadata as importlib_metadata
import json
import os
from pathlib import Path
import subprocess
import sys

REPOSITORY_URL = "YOUR_REPOSITORY_URL"
REPOSITORY_PATH = "YOUR_REPOSITORY_PATH"
repo = Path(REPOSITORY_PATH).expanduser()
if not (repo / "pyproject.toml").exists():
    subprocess.run(["git", "clone", REPOSITORY_URL, str(repo)], check=True)
os.chdir(repo)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[dev]"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[real-model]"], check=True)
for package in ("apertus-eval-prep", "torch", "transformers", "accelerate"):
    print(package, importlib_metadata.version(package))

In [ ]:
from apertus_eval_prep.utils.runtime_profile import profile_runtime
print(json.dumps(profile_runtime(device="auto", precision="auto", quantization="none"), indent=2, default=str))
print("Allocated Colab hardware can differ between sessions; do not treat this profile as production serving hardware.")

In [ ]:
# Edit these values. No access token or credential belongs in this notebook.
MODEL_ID = "YOUR_MODEL_ID"
MODEL_REVISION = "OPTIONAL_PINNED_REVISION"
DEVICE = "auto"
DTYPE = "auto"
QUANTIZATION = "none"
INPUT_PRICE_PER_MILLION = None
OUTPUT_PRICE_PER_MILLION = None
OUTPUT_ROOT = "runs/colab-real"

from apertus_eval_prep.utils.serialization import read_yaml, write_yaml
def render_config(template: str, destination: str) -> str:
    config = read_yaml(template)
    experiment = config.get("experiment")
    run_config = config if isinstance(experiment, dict) is False else experiment["base"]
    adapter = run_config["adapter"]
    adapter.update(model_id=MODEL_ID, revision=MODEL_REVISION)
    params = adapter.setdefault("params", {})
    params.update(tokenizer_id=MODEL_ID, tokenizer_revision=MODEL_REVISION, device=DEVICE, dtype=DTYPE, quantization=QUANTIZATION)
    run_config.setdefault("runtime", {}).update(device=DEVICE, precision=DTYPE, quantization=QUANTIZATION)
    run_config.setdefault("cost", {}).update(input_per_million=INPUT_PRICE_PER_MILLION, output_per_million=OUTPUT_PRICE_PER_MILLION, source="manual_config")
    run_config.setdefault("evidence", {}).update(mode="LOCAL_REAL_MODEL", runtime_environment="google_colab", pricing_source="manual_config")
    if isinstance(experiment, dict):
        experiment["baseline"]["model_revision"] = MODEL_REVISION
        experiment["factors"]["model_revision"] = [MODEL_REVISION]
    write_yaml(destination, config)
    return destination

def platform(*args):
    subprocess.run([sys.executable, "-m", "apertus_eval_prep", *map(str, args)], check=True)

In [ ]:
# 1) Offline framework validation: this must work before any model download.
platform("platform-run", "--config", "configs/platform_smoke.yaml", "--out", "runs/colab-mock-smoke")

# 2) Experimental local real-model smoke evaluation.
smoke_config = render_config("configs/colab/local_real_model_smoke.yaml", "runs/colab-real-smoke.yaml")
platform("platform-run", "--config", smoke_config, "--out", f"{OUTPUT_ROOT}/smoke")

# 3) Four-condition variance analysis (two seeds x two prompt templates).
variance_config = render_config("configs/colab/local_real_model_variance.yaml", "runs/colab-real-variance.yaml")
platform("platform-matrix", "--config", variance_config, "--out", f"{OUTPUT_ROOT}/variance")

# Optional small RAG/agent and sanitized safety evaluations.
RUN_RAG = False
RUN_SAFETY = False
if RUN_RAG:
    rag_config = render_config("configs/colab/local_real_model_rag.yaml", "runs/colab-real-rag.yaml")
    platform("platform-episode", "--config", rag_config, "--out", f"{OUTPUT_ROOT}/rag")
if RUN_SAFETY:
    safety_config = render_config("configs/colab/local_real_model_safety.yaml", "runs/colab-real-safety.yaml")
    platform("platform-safety", "--config", safety_config, "--out", f"{OUTPUT_ROOT}/safety")

In [ ]:
# Ingest completed run directories (never model caches) into Phase 5 points.
# Replace these paths with actual run directories printed by the commands above.
RUNS_TO_COMPARE = ["RUN_DIRECTORY_1", "RUN_DIRECTORY_2"]
points_path = f"{OUTPUT_ROOT}/comparison_points.json"
platform("platform-ingest-runs", "--runs", *RUNS_TO_COMPARE, "--out", points_path)
platform("platform-select", "--points", points_path, "--out", f"{OUTPUT_ROOT}/selection.json")

REPORT_RUN = "RUN_DIRECTORY_1"
platform("platform-report", "--run", REPORT_RUN, "--format", "both", "--out", f"{OUTPUT_ROOT}/reports")

print("Download selected run directories and reports from Colab. Generated runs are git-ignored. Preserve only a deliberately reviewed report for a portfolio case study; do not commit model caches, credentials, or large raw artifacts.")